# 02: Text Embeddings

Convert text to numbers and explore semantic similarity.

## Topics
- One-hot encoding and Bag-of-Words
- Contextual embeddings with Hugging Face
- Cosine similarity
- Visualizing embedding space with PCA/t-SNE

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

print(f"PyTorch version: {torch.__version__}")

## 1. Traditional Text Representations

Before embeddings, text was represented as sparse vectors.

In [ ]:
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are friends"
]

# Build vocabulary
vocab = sorted(set(word for doc in corpus for word in doc.split()))
word_to_idx = {w: i for i, w in enumerate(vocab)}

print(f"Vocabulary ({len(vocab)} words): {vocab}")

In [ ]:
# One-hot encoding
def one_hot_encode(doc, word_to_idx):
    vector = np.zeros(len(word_to_idx))
    for word in doc.split():
        if word in word_to_idx:
            vector[word_to_idx[word]] = 1
    return vector

one_hot_matrix = np.array([one_hot_encode(doc, word_to_idx) for doc in corpus])

print("One-Hot Encoding:")
print(f"Shape: {one_hot_matrix.shape}")
print(f"\nDocument 1 ('{corpus[0]}'):")
for i, (word, vec) in enumerate(zip(vocab, one_hot_matrix[0])):
    if vec == 1:
        print(f"  {word}: {vec:.0f}")

In [ ]:
# Bag-of-Words (count-based)
from collections import Counter

def bag_of_words(doc, word_to_idx):
    vector = np.zeros(len(word_to_idx))
    counts = Counter(doc.split())
    for word, count in counts.items():
        if word in word_to_idx:
            vector[word_to_idx[word]] = count
    return vector

bow_matrix = np.array([bag_of_words(doc, word_to_idx) for doc in corpus])

print("Bag-of-Words Encoding:")
print(f"Shape: {bow_matrix.shape}")
print(f"\nDocument 1 ('{corpus[0]}'):")
for i, (word, count) in enumerate(zip(vocab, bow_matrix[0])):
    if count > 0:
        print(f"  {word}: {count:.0f}")

In [ ]:
# Limitations demo
print("One-Hot / BoW Limitations:")
print("=" * 50)
print(f"Vocabulary size: {len(vocab)} dimensions")
print(f"Sparse vectors (mostly zeros):")
print(f"  Non-zero entries: {np.count_nonzero(one_hot_matrix[0])} / {len(vocab)}")
print(f"\nSemantic meaning: None")
print(f"  'cat' and 'dog' are equally different from 'mat'")
print(f"  No word ordering captured")

## 2. Contextual Embeddings with Hugging Face

Modern embeddings produce dense vectors that capture meaning. The same word gets different vectors in different contexts.

In [ ]:
from transformers import AutoTokenizer, AutoModel

# Load a pre-trained sentence embedding model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

print(f"Model: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def get_embedding(text):
    """Get embedding for a single text."""
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

    # Get model output
    with torch.no_grad():
        outputs = model(**inputs)

    # Mean pooling: average all token embeddings
    token_embeddings = outputs.last_hidden_state
    attention_mask = inputs["attention_mask"].unsqueeze(-1)
    masked_embeddings = token_embeddings * attention_mask
    summed = masked_embeddings.sum(dim=1)
    counts = attention_mask.sum(dim=1)
    mean_pooled = summed / counts

    # Normalize
    normalized = torch.nn.functional.normalize(mean_pooled, p=2, dim=1)
    return normalized.numpy().flatten()


# Test with sample sentences
test_sentences = ["Hello, how are you?", "What is the weather today?"]
for sent in test_sentences:
    emb = get_embedding(sent)
    print(f"'{sent}' -> shape: {emb.shape}, first 5 values: {emb[:5]}")

## 3. Semantic Similarity with Cosine Similarity

Cosine similarity measures the angle between two vectors:
- 1.0 = identical direction (very similar)
- 0.0 = orthogonal (unrelated)
- -1.0 = opposite (dissimilar)

In [ ]:
sentences = [
    "The cat sat on the mat",
    "A feline rested on the rug",
    "The stock market crashed today",
    "Python is a programming language",
    "Dogs are loyal companions",
    "A kitten lay on the carpet"
]

# Generate embeddings
embeddings = np.array([get_embedding(s) for s in sentences])
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# Compute cosine similarity matrix
sim_matrix = cosine_similarity(embeddings)

# Display as heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)

short_labels = [s[:30] + '...' if len(s) > 30 else s for s in sentences]
ax.set_xticks(range(len(sentences)))
ax.set_yticks(range(len(sentences)))
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)

# Add text annotations
for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f'{sim_matrix[i, j]:.2f}', ha='center', va='center', fontsize=9)

plt.colorbar(im, label='Cosine Similarity')
plt.title('Semantic Similarity Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Find most similar pairs
print("Most similar sentence pairs:")
print("=" * 60)

pairs = []
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        pairs.append((sim_matrix[i, j], sentences[i], sentences[j]))

pairs.sort(reverse=True)

for score, s1, s2 in pairs[:5]:
    print(f"\nSimilarity: {score:.4f}")
    print(f"  A: {s1}")
    print(f"  B: {s2}")

## 4. Visualizing the Embedding Space

Reduce high-dimensional embeddings to 2D for visualization using PCA and t-SNE.

In [ ]:
# PCA visualization
pca = PCA(n_components=2)
embeddings_2d_pca = pca.fit_transform(embeddings)

plt.figure(figsize=(10, 8))
colors = plt.cm.Set2(np.linspace(0, 1, len(sentences)))

for i, (sent, color) in enumerate(zip(sentences, colors)):
    plt.scatter(embeddings_2d_pca[i, 0], embeddings_2d_pca[i, 1],
               c=[color], s=150, edgecolors='black', zorder=5)
    plt.annotate(sent[:35] + ('...' if len(sent) > 35 else ''),
                (embeddings_2d_pca[i, 0], embeddings_2d_pca[i, 1]),
                fontsize=9, ha='left', va='bottom',
                xytext=(5, 5), textcoords='offset points')

plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('Embedding Space (PCA)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# t-SNE visualization (better for local structure)
perplexity = min(5, len(sentences) - 1)
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
embeddings_2d_tsne = tsne.fit_transform(embeddings)

plt.figure(figsize=(10, 8))

for i, (sent, color) in enumerate(zip(sentences, colors)):
    plt.scatter(embeddings_2d_tsne[i, 0], embeddings_2d_tsne[i, 1],
               c=[color], s=150, edgecolors='black', zorder=5)
    plt.annotate(sent[:35] + ('...' if len(sent) > 35 else ''),
                (embeddings_2d_tsne[i, 0], embeddings_2d_tsne[i, 1]),
                fontsize=9, ha='left', va='bottom',
                xytext=(5, 5), textcoords='offset points')

plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.title('Embedding Space (t-SNE)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Explore Context-Dependent Embeddings

The word "bank" means different things in different contexts. Contextual embeddings capture this.

In [ ]:
bank_sentences = [
    "I sat on the river bank.",
    "I need to go to the bank.",
    "The bank approved my loan.",
    "She works at a bank downtown.",
]

bank_embeddings = np.array([get_embedding(s) for s in bank_sentences])
bank_sim = cosine_similarity(bank_embeddings)

print("Similarity between 'bank' sentences:")
for i in range(len(bank_sentences)):
    for j in range(i + 1, len(bank_sentences)):
        print(f"  '{bank_sentences[i][:40]}' <-> '{bank_sentences[j][:40]}'")
        print(f"    Similarity: {bank_sim[i, j]:.4f}")

## Key Takeaways

1. **One-hot/BoW** are sparse and lack semantic meaning
2. **Contextual embeddings** capture meaning — similar texts produce similar vectors
3. **Cosine similarity** measures semantic closeness
4. **t-SNE/PCA** let us visualize high-dimensional embedding spaces
5. **Context matters** — the same word gets different embeddings depending on surrounding text